# Carnet de démarrage — sujet 03 · Home Credit

**Objectif du sujet** : décider d'accorder ou de refuser un crédit.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

8,07 % de défauts. Les manquants ne sont pas comblés : c'est une information.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/17v1qEMO8FohlDPU99fAjEcuy_nl-Nim5?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_03_Home_Credit/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Séance 2

| **NOM** | KOONOU SOSSOU REINE |
|---|---|
| **Porteur** | P4 |
|**Etapes assignées**| 2,7|

## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |
| `HomeCredit_columns_description.csv` | entraînement ou annexe |

**Format de sortie attendu** : `SK_ID_CURR,TARGET`.
**Colonne d'identifiant** : `SK_ID_CURR`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [16]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [17]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv', 'HomeCredit_columns_description.csv']
SUJET = "Sujet_03_Home_Credit"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /home/reine/D-p-t-de-groupe-6
fichiers vus : ['CONTRIBUER.md', 'HomeCredit_columns_description.csv', 'README.md', 'a_decider.csv', 'requirements.txt', 'test-collaboration.txt', 'train.csv']


In [23]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import csv
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV avec un cache Parquet complet et une lecture sélective."""
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # Lecture d'un échantillon : pas de cache, on veut seulement explorer.
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    with source.open(newline="", encoding="utf-8-sig") as fichier_csv:
        colonnes_source = next(csv.reader(fichier_csv))

    # Un ancien cache limité à une projection doit être reconstruit avant usage.
    if cache.is_file():
        colonnes_cache = pq.ParquetFile(cache).schema.names
        if set(colonnes_source).issubset(colonnes_cache):
            return pq.ParquetFile(cache).read(columns=colonnes).to_pandas()
        cache.unlink()

    DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
    print(f"construction du cache Parquet : {cache.name} (une seule fois)")
    ecrivain = None
    try:
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
            if ecrivain is None:
                ecrivain = pq.ParquetWriter(cache, lot.schema)
            ecrivain.write_batch(lot)
    finally:
        if ecrivain is not None:
            ecrivain.close()
    return pq.ParquetFile(cache).read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes, complet=True)  # garantit que le cache complet existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux, columns=colonnes):
        yield lot.to_pandas()


In [19]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (246008, 122)
colonnes : 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,123551,0,Cash loans,M,N,Y,0,202500.0,1078200.0,31653.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,400429,0,Cash loans,F,N,Y,2,148500.0,544491.0,20133.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0
2,106912,0,Cash loans,F,Y,N,2,67500.0,363190.5,24268.5,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,3.0
3,383991,0,Revolving loans,F,N,Y,1,144000.0,450000.0,22500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,423763,0,Cash loans,F,N,Y,0,67500.0,675000.0,21775.5,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0


In [20]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    246008.000000
mean          0.080729
std           0.272419
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: TARGET, dtype: float64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [21]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['SK_ID_CURR', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"SK_ID_CURR": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["SK_ID_CURR"], complet=True)
identifiants = a_predire["SK_ID_CURR"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 61503
écrit : soumission.csv · 61,503 lignes · colonnes ['SK_ID_CURR', 'TARGET']


,SK_ID_CURR,TARGET
0,424613,None
1,260084,None
2,214262,None
3,357905,None
4,101573,None
...,...,...
61498,342357,None
61499,382938,None
61500,235122,None
61501,348296,None


In [24]:
# --- 7. Modèle de référence : régression logistique ---------------------------
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# On repart des données complètes : ECHANTILLON ne sert qu'à l'exploration.
donnees = table("train.csv", complet=True)
if donnees[CIBLE].isna().any() or not donnees[CIBLE].isin([0, 1]).all():
    raise ValueError("TARGET doit contenir uniquement des valeurs 0/1 non manquantes.")

y = donnees[CIBLE].astype("int8")
X = donnees.drop(columns=[CIBLE, "SK_ID_CURR"], errors="ignore")
colonnes_numeriques = X.select_dtypes(include=["number", "bool"]).columns.tolist()
colonnes_categorielles = X.columns.difference(colonnes_numeriques).tolist()

# Stratification pour conserver environ 8 % de défauts dans chaque partie.
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pretraitement = ColumnTransformer(
    transformers=[
        (
            "numeriques",
            Pipeline(
                steps=[
                    ("imputation", SimpleImputer(strategy="median", add_indicator=True)),
                    ("normalisation", StandardScaler()),
                ]
            ),
            colonnes_numeriques,
        ),
        (
            "categorielles",
            Pipeline(
                steps=[
                    ("imputation", SimpleImputer(strategy="constant", fill_value="MANQUANT")),
                    ("encodage", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            colonnes_categorielles,
        ),
    ],
    remainder="drop",
)
modele = Pipeline(
    steps=[
        ("pretraitement", pretraitement),
        ("classification", LogisticRegression(C=0.1, max_iter=200)),
    ]
)

# AUC-ROC mesure le classement ; l'average precision est informative en classe rare.
modele.fit(X_train, y_train)
probabilites_validation = modele.predict_proba(X_validation)[:, 1]
print(f"AUC-ROC validation : {roc_auc_score(y_validation, probabilites_validation):.4f}")
print(f"Average precision : {average_precision_score(y_validation, probabilites_validation):.4f}")
print(f"Taux de défaut validation : {y_validation.mean():.2%}")

# Réentraînement sur toutes les lignes avant de prédire les dossiers à décider.
modele.fit(X, y)
a_predire_complet = table(
    "a_decider.csv", colonnes=["SK_ID_CURR", *X.columns], complet=True
)
if a_predire_complet["SK_ID_CURR"].duplicated().any():
    raise ValueError("SK_ID_CURR contient des doublons dans a_decider.csv.")
probabilites = modele.predict_proba(a_predire_complet.loc[:, X.columns])[:, 1]
soumission = ecrire_soumission(
    a_predire_complet["SK_ID_CURR"].to_numpy(),
    probabilites,
    chemin=str(Path.cwd() / "soumission.csv"),
)
soumission.head()

construction du cache Parquet : train.parquet (une seule fois)
AUC-ROC validation : 0.7477
Average precision : 0.2253
Taux de défaut validation : 8.07%
construction du cache Parquet : a_decider.parquet (une seule fois)
écrit : /home/reine/D-p-t-de-groupe-6/D-p-t-de-groupe-6/soumission.csv · 61,503 lignes · colonnes ['SK_ID_CURR', 'TARGET']


,SK_ID_CURR,TARGET
0,424613,0.034466
1,260084,0.035960
2,214262,0.169556
3,357905,0.083920
4,101573,0.076945
